# Building AI News Developer Agent with Google ADK

## 1.1 Getting started

The following python packages will allow to install a handy set of command-line tools that will be useful when working with ADK.

In [ ]:
# !pip install -q google-adk>=1.12.0

In [1]:
from helper import *

# Load environment variables from .env file
load_env()

## 1.2 Setting up the agent

Before we dive into building agents, let's set up a new folder structure with ADK's built-in project scaffolding using the `adk create` command.

When you run `adk create`, it generates three essential files. 
1. The `.env` file securely stores your API credentials and configuration. 
2. The `__init__.py` file marks the directory as a Python package, nabling proper imports. 
3. Most importantly, the `agent.py` file provides a clean foundation where you'll implement your agent.

File structure:
```
app_01/
    __init__.py
    agent.py
    .env
```

ADK create supports two project types. The `--type=code` option generates a Python-based agent in `agent.py`. The `--type=config` option creates a YAML-based agent configuration.

The `--model` parameter specifies the LLM to be used by the agent. We will override this and experiment with different models as we create the agents in this lesson.

### Setting up authentication

Before using any AI models, the first step is to configure API credentials. There are several methods to authenticate with Google's Gemini models, including:
1. Google API key and 
2. Vertex AI based authentication.

For more information on setting up your own authentication, visit [Google](https://cloud.google.com/free?hl=en). 

The `--api_key` parameter automatically configures your project for Google's Gemini API, setting up the proper authentication environment variables. 

Run the cell below to create the folder structure for the agent.

- Method 1  

In [ ]:
# First we create our expected agent folder 
# You can explore available option: !adk create --help 

!adk create --type=code app_01 --model gemini-2.5-flash  --api_key $GEMINI_API_KEY

- Method 2  

On the terminal lanch the following commands 

adk create --type=code app_01

Then   
- open agent.py in the folder app_01 and enclose the required **model gemini-2.0**
- open .env in the folder app_01 and enclose the GEMINI_API_KEY 

## 1.3 Writing the first `agent.py`

 `adk create` command is used to create folders and then write to its `agent.py` using the cell magic in the notebook. Cell magic uses specific commands to interact with the files in your new agent folder. You will use `%%writefile FILENAME` to do this.

Let's start with the absolute simplest agent possible—one that has no internet access at all. You'll create an Agent with a unique name, specify the LLM model, and give it basic instructions.

As with any agent, you need an LLM to start with ADK is model agnostic - meaning you can provide it with any model of your choice like Gemini, Claude, Ollama and even use LiteLLM to bring in other models. Here the Gemini 2.0 flash will be used. 

In [ ]:
%%writefile app_01/agent.py

from google.adk.agents import Agent

root_agent = Agent(
    name="ai_news_agent_simple",
    model="gemini-2.5-flash", 
    instruction="You are an AI News Assistant",
)

Overwriting app_01/agent.py


## 1.5 Adding tools to your agent

**But here's the problem:** try asking this agent about the latest AI developments, and you'll quickly discover it can only tell you about things that happened before its training cutoff date. For an AI news assistant that's supposed to fetch the latest news, that's not particularly helpful.

You need to fix that by providing your agent with **Tools**. Create another empty scaffolding folder to create an agent with a tool by running the cell below.

- Method 1 

In [ ]:
!adk create --type=code app_02 --model gemini-2.5-flash --api_key $GEMINI_API_KEY

- Method 2  

adk create --type=code app_02

Then   
- open agent.py in the folder app_01 and enclose the required **model gemini-2.0-flash**
- open .env in the folder app_01 and enclose the GEMINI_API_KEY 

### Adding Google Search Tool

In this scenario, to fetch the latest news, let's provide your agent with a built-in tool, `google_search`. These built-in tools come pre-packed with the library. 

To add it to the agent, just import it and provide it as a tool in the tools array. Run the cell below to import it.

In [1]:
%%writefile app_02/agent.py

from google.adk.agents import Agent
from google.adk.tools import google_search

root_agent = Agent(
    name="ai_news_agent_simple",
    model="gemini-2.5-flash ", 
    instruction="You are an AI News Assistant. Use Google Search to find recent AI news.",
    tools=[google_search]
)


Overwriting app_02/agent.py


**Refresh the ADK Web UI.** You should see a new app name (app_02) in the dropdown. Select it and start a text conversation with your agent.

Let's test the agent with the Google search tool by asking a query `"What is the latest AI News?"`. Your agent will use the Google Search tool to find current information, process the results, and give you a comprehensive, up-to-date response with sources. Just like that, your agent can now access **real-time information** from across the web!

ADK comes with several other `powerful built-in tools—there` are tools for running your code in a `sandbox`, `querying databases`, even `integrations with Google Workspace tools` like Calendar, Drive etc.

## 1.6 Adding a text model
Text-focused models like `gemini-2.5-flash` are ideal when want optimized text processing. They often provide faster response times. Let's create a variant of the agent using a text-optimized model to see how it behaves differently.

Create a new folder for the text-based agent by running the cell below.

In [ ]:
!adk create --type=code app_03 --model gemini-2.5-flash --api_key $GEMINI_API_KEY

Run the below cell to create a new agent. 

In [2]:
%%writefile app_03/agent.py

from google.adk.agents import Agent
from google.adk.tools import google_search

root_agent = Agent(
    name="ai_news_agent_simple",
    model="gemini-2.5-flash",
    instruction="You are an AI News Assistant. Use Google Search to find recent AI news.",
    tools=[google_search]
)


Overwriting app_03/agent.py


**Refresh the ADK Web UI**, and select the app_03 from the dropdown menu. Chat with your agent using text and ask it: "Give me the latest AI news".

*Bonus:* Explore the different capabilities in the `Web UI` like the `Events tab`.

## 1.8 Fine-tuning agent instructions

So far your agent has simple instructions, but for reliable behavior, you need more sophisticated instruction engineering. Let's enhance your agent with strict behavioral controls.

Create a new folder using the `adk create` command.

In [ ]:
# You can also modify the instructions  

!adk create --type=code app_05 --model gemini-2.5-flash --api_key $GEMINI_API_KEY

In [ ]:
adk create --type=code app_05

Write the updated agent code to the `agent.py` file.

In [4]:
%%writefile app_05/agent.py

from google.adk.agents import Agent
from google.adk.tools import google_search

root_agent = Agent(
    name="ai_news_agent_strict",
    model="gemini-2.5-flash",
    instruction="""
    **Your Core Identity and Sole Purpose:**
    You are a specialized AI News Assistant. Your sole and exclusive purpose is to find and summarize recent news (from the last few weeks) about Artificial Intelligence.

    **Strict Refusal Mandate:**
    If a user asks about ANY topic that is not recent AI news, you MUST refuse.
    For off-topic requests, respond with the exact phrase: "Sorry, I can't answer anything about this. I am only supposed to answer about the latest AI news."

    **Required Workflow for Valid Requests:**
    1. You MUST use the `google_search` tool to find information.
    2. You MUST base your answer strictly on the search results.
    3. You MUST cite your sources.
    """,
    tools=[google_search]
)

Overwriting app_05/agent.py


### Adding the Python code executor Tool

To fetch the latest news, the agent has been provided with a built-in tool, **google_search**. These built-in tools come `pre-packed with the library`. Now, in order to allow the agent also to execute code and debug using Gemini models, it has been used the **built_in_code_execution tool** that enables the agent to execute code, specifically when using Gemini 2 and higher models. This allows the model to perform tasks like calculations, data manipulation, or running small scripts. Also this tool comes from `pre-packed with the library`. To add it to the agent, just import it and provide it as a tool in the tools array. Run the cell below to import it

In [ ]:
# Setting up the  agent. The first step is to create the ADK folder
# You can explore available option: !adk create --help 

!adk create --type=code app06 --model gemini-2.5-flash --api_key $GEMINI_API_KEY

In [1]:
%%writefile app_08/agent.py
from google.adk.agents import Agent
from google.adk.tools import google_search
from google.adk.tools.agent_tool import AgentTool
from google.adk.code_executors import BuiltInCodeExecutor


# --------------------------------------------------
# Search Agent
# --------------------------------------------------

search_agent = Agent(
    model="gemini-2.5-flash",
    name="SearchAgent",
    description="Specialist agent for Google Search.",
    instruction="""
You are a specialist in Google Search.

- Use google_search to find information on the web
- Return concise, factual results
""",
    tools=[google_search],
)


# --------------------------------------------------
# Code Execution Agent
# --------------------------------------------------

coding_agent = Agent(
    model="gemini-2.5-flash",
    name="CodeAgent",
    description="Specialist agent for Python code execution.",
    instruction="""
You are a specialist in Python code execution.

- Execute Python code when requested
- Return the execution result only
""",
    code_executor=BuiltInCodeExecutor(),
)


# --------------------------------------------------
# Root Routing Agent
# --------------------------------------------------

root_agent = Agent(
    name="RootAgent",
    model="gemini-2.5-flash",
    description="Root routing agent that delegates to specialist agents.",
    instruction="""
You are a routing agent.

Routing rules:
- Use SearchAgent for search or news-related requests
- Use CodeAgent for Python code execution
- Do NOT answer directly
""",
    tools=[
        AgentTool(agent=search_agent),
        AgentTool(agent=coding_agent),
    ],
)


Overwriting app_08/agent.py


In [2]:
%%writefile app_07/agent.py
from google.adk.agents.llm_agent import Agent
from google.adk.tools import google_search

root_agent = Agent(
    model="gemini-2.5-flash",
    name="root_agent",
    description="Analyzes recent AI-related news for US-listed companies using Google Search.",
    instruction="""You are an AI News Analyst specializing in recent AI news about US-listed companies.
Your primary goal is to be interactive, concise, and transparent about your information sources.

RECENT means news published within the past 7 days unless the user specifies otherwise.

Your Workflow:

1. Clarify First:
If the user makes a general request for news (e.g., "give me AI news"), your very first response MUST be:
"Sure, I can do that. How many news items would you like me to find?"
Wait for their answer before doing anything else.

2. Search and Enrich:
Once the user specifies a number:
- Use the `google_search` tool to find the requested number of recent AI news articles.
- For each article:
  - Identify the US-listed company mentioned.
  - Identify its stock ticker.
  - If the stock ticker cannot be confidently determined, write "(ticker unavailable)".
  - Ignore articles that do not clearly involve a US-listed company.

3. Present Headlines with Citations:
Display the results as a concise, numbered list.

You MUST:
- Start with: "Using `google_search` for news, here are the top headlines:"
- Include the company name and stock ticker (or fallback) with each headline.
- Explicitly mention `google_search` when presenting news.

Format:
1. [Headline] – [Company Name] ([Ticker])
2. [Headline] – [Company Name] ([Ticker])

4. Engage and Wait:
After presenting the headlines, ask:
"Which of these are you interested in? Or should I search for more?"

5. Discuss One Topic:
If the user selects a headline:
- Provide a detailed summary for ONLY that single item.
- Mention that the information was found using `google_search`.
- End by handing the conversation back to the user.

Strict Rules:
- Stay on Topic: You ONLY discuss AI-related news for US-listed companies.
  If asked anything else, respond:
  "I can only provide recent AI news for US-listed companies."
- Short Turns: Keep responses brief and avoid long monologues.
- No Guessing: Never invent stock tickers or companies.
- Cite Tools: Always mention `google_search` when presenting or summarizing news.
""",
    tools=[google_search],
)


Overwriting app_07/agent.py


### Understanding Function Tools
What makes a good function tool:

- **Clear Documentation**: Comprehensive docstrings that ADK uses for tool descriptions  
- **Type Annotations**: Essential for ADK's automatic tool registration and schema validation  
- **Robust Error Handling**: Graceful failure modes prevent agent crashes  
- **Consistent Return Format**: Predictable output structure agents can reliably process  

## 3.4 Adding your root agent 
Now let's create your agent with sophisticated instructions that use both the Google Search tool and your custom financial tool.

Let's examine the enhanced instructions:

### Key Instruction Patterns
Notice how these instructions implement several best practices:

1. **Structured Workflow**: 5-step process from clarification to detailed discussion  
2. **Tool Citation Requirements**: Agent must cite google_search and get_financial_context usage  
3. **Interactive Design**: Prompts user for input at each stage rather than providing monologues  
4. **Scope Boundaries**: Clear rules about staying focused on AI news for US-listed companies  
5. **Error Handling**: Graceful responses when asked about off-topic subjects  

This creates a much more reliable and user-friendly conversational experience.

## 3.5 Test Your Agent
Let's test your agent using the ADK Web UI. These steps will be repeated for all lessons in this course.

1. Run the below cell to start a new terminal.
2. In the terminal, navigate to the L3 directory using the following command `cd L3`
3. In the terminal, start the ADK web server using the command `adk web --host 0.0.0.0 --port 8003`

### Try This Testing Sequence
Once your agent is running, try this conversation flow to see all the features in action:

1. Start with a general request: "Give me AI news for Google"  
2. Respond to clarification: "Give me 3 top AI news items for publicly traded US tech stocks"  
3. Pick a specific story: "Tell me more about the first one"  
4. Test boundaries: "What's the weather today?" (should be refused)  

Watch how the agent uses both tools in coordination and maintains conversation flow.

##  Expand Your Tool
Now it's your turn to experiment and enhance the agent! Try these modifications:



### Exercise 2: Instruction Refinement
Enhance the agent instructions to:

- Request user preference for number of companies to track (1-5 range)  
- Include timestamp information about when the data was fetched

### Exercise 3: Conversation Flow
Test the conversation boundaries:

- Test with different types of AI news (research, products, acquisitions)
- See how the agent handles follow-up questions about specific companies

### Exercise 4: Error Handling
Test the tool's error handling:

- See how the agent handles network connectivity issues  

In [ ]:
from google.adk.tools.agent_tool import AgentTool
from google.adk.agents import Agent
from google.adk.tools import google_search
from google.adk.code_executors import BuiltInCodeExecutor

search_agent = Agent(
    #model='gemini-2.0-flash',
    model='gemini-2.5-flash',
    name='SearchAgent',
    instruction="""
    You're a specialist in Google Search
    """,
    tools=[google_search],
)
coding_agent = Agent(
    #model='gemini-2.0-flash',
    model='gemini-2.5-flash',
    name='CodeAgent',
    instruction="""
    You're a specialist in Code Execution
    """,
    code_executor=BuiltInCodeExecutor(),
)
root_agent = Agent(
    name="RootAgent",
    model="gemini-2.5-flash",
    #model='gemini-2.0-flash',
    description="Root Agent",
    tools=[AgentTool(agent=search_agent), AgentTool(agent=coding_agent)],
)

In [1]:
%%writefile app_10/agent.py

from google.adk.agents import Agent
from google.adk.tools import google_search
from google.adk.tools.agent_tool import AgentTool
from google.adk.code_executors import BuiltInCodeExecutor
from adk.web import WebApp
import re

# ----------------------------
# AI Developer News Agent
# ----------------------------
search_agent = Agent(
    model="gemini-2.5-flash",
    name="AIDevSearchAgent",
    description="Specialist agent for AI news relevant to developers.",
    instruction="""
You are an AI News Analyst for developers.

Workflow:

1. Clarify First:
- If user requests general AI news, respond:
  "Sure, I can do that. How many news items would you like me to find?"
- Wait for user's answer before searching.

2. Search and Enrich:
- Use `google_search` to find recent AI news.
- Focus on AI articles, platforms, and use cases.
- Ignore news not relevant to developers.

3. Present Headlines:
- Numbered list: 1. [Headline] – [Topic/Platform/Use Case]
- Start with: "Using `google_search` for news, here are the top headlines:"
- Cite `google_search`.

4. Engage and Wait:
- Ask: "Which of these are you interested in? Or should I search for more?"

5. Discuss One Topic:
- Provide a detailed summary of the selected headline.
- Cite `google_search`.
- Hand conversation back to user.
""",
    tools=[google_search],
)

# ----------------------------
# Python Code Execution Agent
# ----------------------------
coding_agent = Agent(
    model="gemini-2.5-flash",
    name="CodeAgent",
    description="Specialist agent for Python code execution.",
    instruction="""
You are a specialist in Python code execution.
- Execute Python code when requested
- Return the execution result only
""",
    code_executor=BuiltInCodeExecutor(),
)

# ----------------------------
# Root Routing Agent
# ----------------------------
root_agent = Agent(
    name="RootAgent",
    model="gemini-2.5-flash",
    description="Root routing agent that delegates to specialist agents.",
    instruction="""
You are a routing agent.
- Use AIDevSearchAgent for AI developer news requests
- Use CodeAgent for Python code execution
- Do NOT answer directly
""",
    tools=[
        AgentTool(agent=search_agent),
        AgentTool(agent=coding_agent),
    ],
)

# ----------------------------
# Helper Functions
# ----------------------------
def extract_headlines(response_text):
    """
    Parses numbered headlines from agent response.
    Assumes format: 1. Headline – Topic
    """
    matches = re.findall(r"\d+\.\s*(.+?)\s*–", response_text)
    return matches

def handle_user_input(user_input: str, session_state: dict):
    """
    Handles a single user message for ADK Web.
    session_state keeps track of headlines and other session data.
    """
    if session_state is None:
        session_state = {"headlines": []}

    response_text = ""
    headlines = session_state.get("headlines", [])

    # Exit command
    if user_input.lower() in {"exit", "quit"}:
        response_text = "Goodbye!"
        session_state["headlines"] = []
        return response_text, session_state

    # Execute Python code
    if user_input.lower().startswith("execute python code:"):
        code_to_run = user_input[len("execute python code:"):].strip()
        try:
            result = coding_agent.run(code_to_run)
            response_text = f"**Code Result:**\n{result}"
        except Exception as e:
            response_text = f"Error executing code: {e}"
        return response_text, session_state

    # User selects a headline number
    if user_input.isdigit() and headlines:
        idx = int(user_input) - 1
        if 0 <= idx < len(headlines):
            headline = headlines[idx]
            response_text = f"Summarizing the selected news item using `google_search`...\n"
            summary = search_agent.run(f"Provide a detailed summary for: {headline}")
            response_text += summary
        else:
            response_text = "Invalid selection. Please choose a valid number."
        return response_text, session_state

    # User asks for more news
    if user_input.lower() in {"more", "search more"} and headlines:
        response_text = "Searching for more news using `google_search`...\n"
        response = root_agent.run("Find more AI news for developers")
        response_text += response
        session_state["headlines"] = extract_headlines(response)
        return response_text, session_state

    # General request routed to RootAgent
    response = root_agent.run(user_input)
    response_text = response
    session_state["headlines"] = extract_headlines(response)
    return response_text, session_state




Overwriting app_10/agent.py


- Third-party tools : Hugging Face  
Access models, datasets, research papers, and AI tools. 

In [ ]:
%%writefile app_11/agent.py

from google.adk.agents import Agent
from google.adk.tools.mcp_tool import McpToolset
from google.adk.tools.mcp_tool.mcp_session_manager import StdioConnectionParams
from mcp import StdioServerParameters

HUGGING_FACE_TOKEN = "YOUR HUGGING FACE TOKEN HERE"

root_agent = Agent(
    #model="gemini-2.5-pro",
    model="gemini-2.5-flash",
    name="hugging_face_agent",
    instruction="Help users get information from Hugging Face",
    tools=[
        McpToolset(
            connection_params=StdioConnectionParams(
                server_params = StdioServerParameters(
                    command="npx",
                    args=[
                        "-y",
                        "@llmindset/hf-mcp-server",
                    ],
                    env={
                        "HF_TOKEN": HUGGING_FACE_TOKEN,
                    }
                ),
                timeout=30,
            ),
        )
    ],
)

Overwriting app_11/agent.py


- Third-party tools : GitHub 
Analyze code, mamage issues and PRs, and autimate workflows  

In [ ]:
%%writefile app_12/agent.py

from google.adk.agents import Agent
from google.adk.tools.mcp_tool import McpToolset
from google.adk.tools.mcp_tool.mcp_session_manager import StreamableHTTPServerParams

GITHUB_TOKEN = "YOUR GITHUB TOKEN HERE"

root_agent = Agent(
    model="gemini-2.5-pro",
    name="github_agent",
    instruction="Help users get information from GitHub",
    tools=[
        McpToolset(
            connection_params=StreamableHTTPServerParams(
                url="https://api.githubcopilot.com/mcp/",
                headers={
                    "Authorization": f"Bearer {GITHUB_TOKEN}",
                    "X-MCP-Toolsets": "all",
                    "X-MCP-Readonly": "true"
                },
            ),
        )
    ],
)

Overwriting app_12/agent.py


-START FROM HERE 

In [ ]:
#Method 1 
#!adk create --type=code app_01 --model gemini-2.5-flash --api_key $GEMINI_API_KEY

- Method 2  

On the terminal lanch the following commands  
```bash   
adk create --type=code app_01  
```
Then   
- open agent.py in the folder app_01 and enclose the required **model gemini-2.5-flash**
- open .env in the folder app_01 and enclose the GEMINI_API_KEY 

In [ ]:
%%writefile app_001/agent.py

from google.adk.agents import Agent
from google.adk.tools import google_search
from google.adk.tools.agent_tool import AgentTool
from google.adk.code_executors import BuiltInCodeExecutor
from adk.web import WebApp
import re

# ----------------------------
# AI Developer News Agent
# ----------------------------
search_agent = Agent(
    model="gemini-2.5-flash",
    name="AIDevSearchAgent",
    description="Specialist agent for AI news relevant to developers.",
    instruction="""
You are an AI News Analyst for developers.

Workflow:

1. Clarify First:
- If user requests general AI news, respond:
  "Sure, I can do that. How many news items would you like me to find?"
- Wait for user's answer before searching.

2. Search and Enrich:
- Use `google_search` to find recent AI news.
- Focus on AI articles, platforms, and use cases.
- Ignore news not relevant to developers.

3. Present Headlines:
- Numbered list: 1. [Headline] – [Topic/Platform/Use Case]
- Start with: "Using `google_search` for news, here are the top headlines:"
- Cite `google_search`.

4. Engage and Wait:
- Ask: "Which of these are you interested in? Or should I search for more?"

5. Discuss One Topic:
- Provide a detailed summary of the selected headline.
- Cite `google_search`.
- Hand conversation back to user.
""",
    tools=[google_search],
)

# ----------------------------
# Python Code Execution Agent
# ----------------------------
coding_agent = Agent(
    model="gemini-2.5-flash",
    name="CodeAgent",
    description="Specialist agent for Python code execution.",
    instruction="""
You are a specialist in Python code execution.
- Execute Python code when requested
- Return the execution result only
""",
    code_executor=BuiltInCodeExecutor(),
)

# ----------------------------
# Root Routing Agent
# ----------------------------
root_agent = Agent(
    name="RootAgent",
    model="gemini-2.5-flash",
    description="Root routing agent that delegates to specialist agents.",
    instruction="""
You are a routing agent.
- Use AIDevSearchAgent for AI developer news requests
- Use CodeAgent for Python code execution
- Do NOT answer directly
""",
    tools=[
        AgentTool(agent=search_agent),
        AgentTool(agent=coding_agent),
    ],
)

# ----------------------------
# Helper Functions
# ----------------------------
def extract_headlines(response_text):
    """
    Parses numbered headlines from agent response.
    Assumes format: 1. Headline – Topic
    """
    matches = re.findall(r"\d+\.\s*(.+?)\s*–", response_text)
    return matches

def handle_user_input(user_input: str, session_state: dict):
    """
    Handles a single user message for ADK Web.
    session_state keeps track of headlines and other session data.
    """
    if session_state is None:
        session_state = {"headlines": []}

    response_text = ""
    headlines = session_state.get("headlines", [])

    # Exit command
    if user_input.lower() in {"exit", "quit"}:
        response_text = "Goodbye!"
        session_state["headlines"] = []
        return response_text, session_state

    # Execute Python code
    if user_input.lower().startswith("execute python code:"):
        code_to_run = user_input[len("execute python code:"):].strip()
        try:
            result = coding_agent.run(code_to_run)
            response_text = f"**Code Result:**\n{result}"
        except Exception as e:
            response_text = f"Error executing code: {e}"
        return response_text, session_state

    # User selects a headline number
    if user_input.isdigit() and headlines:
        idx = int(user_input) - 1
        if 0 <= idx < len(headlines):
            headline = headlines[idx]
            response_text = f"Summarizing the selected news item using `google_search`...\n"
            summary = search_agent.run(f"Provide a detailed summary for: {headline}")
            response_text += summary
        else:
            response_text = "Invalid selection. Please choose a valid number."
        return response_text, session_state

    # User asks for more news
    if user_input.lower() in {"more", "search more"} and headlines:
        response_text = "Searching for more news using `google_search`...\n"
        response = root_agent.run("Find more AI news for developers")
        response_text += response
        session_state["headlines"] = extract_headlines(response)
        return response_text, session_state

    # General request routed to RootAgent
    response = root_agent.run(user_input)
    response_text = response
    session_state["headlines"] = extract_headlines(response)
    return response_text, session_state


In [ ]:
!adk create --type=code app_02 --model gemini-2.5-flash --api_key $GEMINI_API_KEY

In [ ]:
%%writefile app_002/agent.py


from google.adk.agents import Agent
from google.adk.tools import google_search
from google.adk.tools.agent_tool import AgentTool
from google.adk.code_executors import BuiltInCodeExecutor
from adk.web import WebApp
import re

# ----------------------------
# AI Developer News Agent
# ----------------------------
search_agent = Agent(
    model="gemini-2.5-flash",
    name="AIDevSearchAgent",
    description="Specialist agent for AI news relevant to developers.",
    instruction="""
You are an AI News Analyst for developers.

Workflow:

1. Clarify First:
- If user requests general AI news, respond:
  "Sure, I can do that. How many news items would you like me to find?"
- Wait for user's answer before searching.

2. Search and Enrich:
- Use `google_search` to find recent AI news.
- Focus on AI articles, platforms, and use cases.
- Ignore news not relevant to developers.

3. Present Headlines:
- Numbered list: 1. [Headline] – [Topic/Platform/Use Case]
- Start with: "Using `google_search` for news, here are the top headlines:"
- Cite `google_search`.

4. Engage and Wait:
- Ask: "Which of these are you interested in? Or should I search for more?"

5. Discuss One Topic:
- Provide a detailed summary of the selected headline.
- Cite `google_search`.
- Hand conversation back to user.
""",
    tools=[google_search],
)

# ----------------------------
# Python Code Execution Agent
# ----------------------------
coding_agent = Agent(
    model="gemini-2.5-flash",
    name="CodeAgent",
    description="Specialist agent for Python code execution.",
    instruction="""
You are a specialist in Python code execution.
- Execute Python code when requested
- Return the execution result only
""",
    code_executor=BuiltInCodeExecutor(),
)

# ----------------------------
# HuggingFace Execution Agent
# ----------------------------
from google.adk.agents import Agent
from google.adk.tools.mcp_tool import McpToolset
from google.adk.tools.mcp_tool.mcp_session_manager import StdioConnectionParams
from mcp import StdioServerParameters

HUGGING_FACE_TOKEN = "YOUR HUGGING FACE TOKEN HERE"

hf_agent = Agent(
    model="gemini-2.5-flash",
    name="hugging_face_agent",
    instruction="Help users get information from Hugging Face",
    tools=[
        McpToolset(
            connection_params=StdioConnectionParams(
                server_params = StdioServerParameters(
                    command="npx",
                    args=[
                        "-y",
                        "@llmindset/hf-mcp-server",
                    ],
                    env={
                        "HF_TOKEN": HUGGING_FACE_TOKEN,
                    }
                ),
                timeout=30,
            ),
        )
    ],
)


# ----------------------------
# GitHub Execution Agent
# ----------------------------
from google.adk.agents import Agent
from google.adk.tools.mcp_tool import McpToolset
from google.adk.tools.mcp_tool.mcp_session_manager import StreamableHTTPServerParams

GITHUB_TOKEN = "YOUR GITHUB TOKEN HERE"

git_agent = Agent(
    model="gemini-2.5-flash",
    name="github_agent",
    instruction="Help users get information from GitHub",
    tools=[
        McpToolset(
            connection_params=StreamableHTTPServerParams(
                url="https://api.githubcopilot.com/mcp/",
                headers={
                    "Authorization": f"Bearer {GITHUB_TOKEN}",
                    "X-MCP-Toolsets": "all",
                    "X-MCP-Readonly": "true"
                },
            ),
        )
    ],
)



# ----------------------------
# Root Routing Agent
# ----------------------------
root_agent = Agent(
    name="RootAgent",
    model="gemini-2.5-flash",
    description="Root routing agent that delegates to specialist agents.",
    instruction="""
You are a routing agent.
- Use AIDevSearchAgent for AI developer news requests
- Use CodeAgent for Python code execution
- Do NOT answer directly
""",
    tools=[
        AgentTool(agent=search_agent),
        AgentTool(agent=coding_agent),
        AgentTool(agent=hf_agent),
        AgentTool(agent=git_agent),
    ],
)

# ----------------------------
# Helper Functions
# ----------------------------
def extract_headlines(response_text):
    """
    Parses numbered headlines from agent response.
    Assumes format: 1. Headline – Topic
    """
    matches = re.findall(r"\d+\.\s*(.+?)\s*–", response_text)
    return matches

def handle_user_input(user_input: str, session_state: dict):
    """
    Handles a single user message for ADK Web.
    session_state keeps track of headlines and other session data.
    """
    if session_state is None:
        session_state = {"headlines": []}

    response_text = ""
    headlines = session_state.get("headlines", [])

    # Exit command
    if user_input.lower() in {"exit", "quit"}:
        response_text = "Goodbye!"
        session_state["headlines"] = []
        return response_text, session_state

    # Execute Python code
    if user_input.lower().startswith("execute python code:"):
        code_to_run = user_input[len("execute python code:"):].strip()
        try:
            result = coding_agent.run(code_to_run)
            response_text = f"**Code Result:**\n{result}"
        except Exception as e:
            response_text = f"Error executing code: {e}"
        return response_text, session_state

    # User selects a headline number
    if user_input.isdigit() and headlines:
        idx = int(user_input) - 1
        if 0 <= idx < len(headlines):
            headline = headlines[idx]
            response_text = f"Summarizing the selected news item using `google_search`...\n"
            summary = search_agent.run(f"Provide a detailed summary for: {headline}")
            response_text += summary
        else:
            response_text = "Invalid selection. Please choose a valid number."
        return response_text, session_state

    # User asks for more news
    if user_input.lower() in {"more", "search more"} and headlines:
        response_text = "Searching for more news using `google_search`...\n"
        response = root_agent.run("Find more AI news for developers")
        response_text += response
        session_state["headlines"] = extract_headlines(response)
        return response_text, session_state

    # General request routed to RootAgent
    response = root_agent.run(user_input)
    response_text = response
    session_state["headlines"] = extract_headlines(response)
    return response_text, session_state


